# The Muon Optimizer — Deep Dive

The Parameter Golf baseline uses a custom optimizer called **Muon** for matrix-shaped parameters (attention projections, MLP weights). This notebook explores what it does, why it works, and how it differs from Adam.

## TL;DR
- Muon applies SGD with momentum, then **orthogonalizes** the update via Newton-Schulz iteration
- This means every update is a rotation/reflection — it preserves the scale of weight matrices
- Combined with separate Adam optimizers for embeddings and scalar parameters
- Result: faster convergence than pure Adam on this type of small model

## References
- [Muon blog post](https://kellerjordan.github.io/posts/muon/) by Keller Jordan
- The baseline uses it at `parameter-golf/train_gpt_mlx.py` lines 457-482

In [ ]:
import sys
sys.path.insert(0, '../parameter-golf')

import numpy as np
import mlx.core as mx
from train_gpt_mlx import zeropower_newtonschulz5, rms_norm

## 1. What is Newton-Schulz Orthogonalization?

The core of Muon is `zeropower_newtonschulz5()` — it takes a matrix and finds the nearest orthogonal matrix.

An **orthogonal matrix** Q satisfies: Q @ Q.T = I (identity)

Why? Because orthogonal updates:
- Don't change the scale of weight matrices (no exploding/vanishing gradients)
- Explore the parameter space more uniformly
- Act like a "rotation" rather than a "stretch" of the weights

The Newton-Schulz iteration is an iterative algorithm that converges to the nearest orthogonal matrix in just 5 steps.

In [ ]:
# Let's watch Newton-Schulz in action
# Start with a random gradient matrix
G = mx.random.normal((64, 32))

print(f"Input gradient shape: {G.shape}")
print(f"Input Frobenius norm: {float(mx.sqrt(mx.sum(G * G))):.4f}")

# Apply Newton-Schulz with different step counts to see convergence
for n_steps in [1, 2, 3, 5, 10]:
    Q = zeropower_newtonschulz5(G, n_steps)
    # Check orthogonality: Q.T @ Q should be close to identity
    QtQ = np.array((Q.astype(mx.float32).T @ Q.astype(mx.float32)))
    I = np.eye(QtQ.shape[0])
    ortho_error = np.max(np.abs(QtQ - I))
    print(f"  {n_steps} steps: max|Q'Q - I| = {ortho_error:.6f}  (0 = perfectly orthogonal)")

print(f"\nThe baseline uses 5 steps — good convergence!")

In [ ]:
# Visualize what Newton-Schulz does step by step
G = mx.random.normal((8, 8))  # small matrix for visualization
a, b, c = 3.4445, -4.7750, 2.0315  # magic constants

x = G.astype(mx.float32)
x = x / (mx.sqrt(mx.sum(x * x)) + 1e-7)

print("Newton-Schulz iteration (8x8 matrix):")
print(f"  Step 0: singular values = {np.linalg.svd(np.array(x), compute_uv=False).round(3)}")

for step in range(5):
    a_mat = x @ x.T
    b_mat = b * a_mat + c * (a_mat @ a_mat)
    x = a * x + b_mat @ x
    svs = np.linalg.svd(np.array(x.astype(mx.float32)), compute_uv=False)
    print(f"  Step {step+1}: singular values = {svs.round(3)}")

print(f"\nAll singular values converge to 1.0 — that's what 'orthogonal' means!")
print(f"The matrix becomes a pure rotation/reflection, no scaling.")

## 2. The Muon Update Rule

For each matrix parameter, Muon does:

```python
buf = momentum * buf + gradient          # SGD momentum
g_eff = gradient + momentum * buf        # Nesterov-like
g_ortho = orthogonalize(g_eff)           # Newton-Schulz
scale = sqrt(max(1, rows/cols))          # Scale factor for non-square matrices
param = param - lr * g_ortho * scale     # Update
```

Key differences from Adam:
1. **No per-parameter adaptive learning rate** — just a global LR
2. **No second moment** — no v (variance) term
3. **Orthogonal updates** — the direction is constrained
4. **SGD momentum** (not EMA) — simpler accumulation

In [ ]:
# Simulate a Muon update vs Adam update
W = mx.random.normal((64, 64)) * 0.02  # weight matrix
G = mx.random.normal((64, 64)) * 0.01  # gradient

# Muon update
lr_muon = 0.04
G_ortho = zeropower_newtonschulz5(G, 5)
scale = (max(1.0, 64/64)) ** 0.5  # = 1.0 for square
W_muon = W - lr_muon * G_ortho * scale

# Compare norms
print(f"Original W norm:    {float(mx.sqrt(mx.sum(W * W))):.4f}")
print(f"After Muon update:  {float(mx.sqrt(mx.sum(W_muon * W_muon))):.4f}")

# Plain SGD for comparison
lr_sgd = 0.04
W_sgd = W - lr_sgd * G
print(f"After SGD update:   {float(mx.sqrt(mx.sum(W_sgd * W_sgd))):.4f}")

print(f"\nMuon's orthogonal update preserves the weight scale better.")
print(f"SGD's update can arbitrarily change the scale.")

## 3. The Three-Optimizer Split

The baseline doesn't use Muon for everything. It splits parameters into three groups:

| Group | Optimizer | LR | Parameters |
|-------|-----------|-----|------------|
| **Matrices** | Muon | 0.04 | Attention Q/K/V/proj, MLP fc/proj (2D, ≥65K params) |
| **Embeddings** | Adam | 0.05 | Token embedding (tied with output head) |
| **Scalars** | Adam | 0.04 | attn_scale, mlp_scale, resid_mix, q_gain, skip_weights |

Why? Muon's orthogonal updates only make sense for 2D matrices. Vectors and scalars need standard Adam.

In [ ]:
from train_gpt_mlx import Hyperparameters, GPT, SplitOptimizers, CONTROL_TENSOR_NAME_PATTERNS
from mlx.utils import tree_flatten

args = Hyperparameters()
mx.random.seed(1337)
model = GPT(
    vocab_size=args.vocab_size, num_layers=args.num_layers, dim=args.model_dim,
    num_heads=args.num_heads, num_kv_heads=args.num_kv_heads, mlp_mult=args.mlp_mult,
    logit_chunk_tokens=0, logit_softcap=args.logit_softcap, rope_base=args.rope_base,
    tied_embed_init_std=args.tied_embed_init_std, qk_gain_init=args.qk_gain_init,
)
opt = SplitOptimizers(model, args)

params = dict(tree_flatten(model.parameters()))

print("=== Optimizer Groups ===")
print(f"\nMuon (matrix parameters): {len(opt.matrix_keys)} tensors")
muon_params = sum(int(np.prod(params[k].shape)) for k in opt.matrix_keys)
for k in opt.matrix_keys[:6]:  # Show first few
    print(f"  {k}: {params[k].shape}")
if len(opt.matrix_keys) > 6:
    print(f"  ... and {len(opt.matrix_keys) - 6} more")
print(f"  Total: {muon_params:,} params")

print(f"\nAdam (embedding): 1 tensor")
embed_params = int(np.prod(params[opt.embed_key].shape))
print(f"  {opt.embed_key}: {params[opt.embed_key].shape} ({embed_params:,} params)")

print(f"\nAdam (scalar parameters): {len(opt.scalar_keys)} tensors")
scalar_params = sum(int(np.prod(params[k].shape)) for k in opt.scalar_keys)
for k in opt.scalar_keys[:6]:
    print(f"  {k}: {params[k].shape}")
if len(opt.scalar_keys) > 6:
    print(f"  ... and {len(opt.scalar_keys) - 6} more")
print(f"  Total: {scalar_params:,} params")

total = muon_params + embed_params + scalar_params
print(f"\n--- Summary ---")
print(f"Muon:      {muon_params:>10,} params ({muon_params/total*100:.1f}%)")
print(f"Adam-emb:  {embed_params:>10,} params ({embed_params/total*100:.1f}%)")
print(f"Adam-scal: {scalar_params:>10,} params ({scalar_params/total*100:.1f}%)")
print(f"Total:     {total:>10,} params")

## 4. Momentum Warmup

Muon's momentum starts at 0.85 and warms up to 0.95 over 500 steps.
This prevents early training instability — high momentum early on would amplify noisy gradients.

In [ ]:
# Visualize momentum warmup
warmup_steps = args.muon_momentum_warmup_steps
start_mom = args.muon_momentum_warmup_start
end_mom = args.muon_momentum

steps = list(range(1000))
momentums = []
for s in steps:
    t = min(s / warmup_steps, 1.0)
    m = (1.0 - t) * start_mom + t * end_mom
    momentums.append(m)

print(f"Momentum warmup: {start_mom} → {end_mom} over {warmup_steps} steps")
print(f"\nStep   0: momentum = {momentums[0]:.4f}")
print(f"Step 100: momentum = {momentums[100]:.4f}")
print(f"Step 250: momentum = {momentums[250]:.4f}")
print(f"Step 500: momentum = {momentums[500]:.4f}")
print(f"Step 999: momentum = {momentums[999]:.4f} (stays at target after warmup)")

## Key Takeaways

1. **Muon = SGD momentum + orthogonalization** — simpler than Adam but effective for matrices
2. **Newton-Schulz** converges in ~5 iterations — cheap to compute
3. **Orthogonal updates preserve scale** — no exploding/vanishing weight norms
4. **Three-way split** is necessary — Muon only works for 2D matrices
5. **Momentum warmup** prevents early instability

### Potential experiments
- What happens with different momentum values? (top submissions use 0.92→0.99)
- More Newton-Schulz steps? (current: 5)
- Different learning rates per layer?
- Muon with weight decay? (some submissions add decoupled WD of 0.04)